# Kerswell horizontal Coulomb dam break — AVAC

This notebook tests Coulomb spreading and natural arrest against Kerswell's analytical solution. It evaluates both moving boundaries and depth profiles.


## Reproducible environment

The first code cell installs the repository's validation package and Python dependencies into the active kernel. A clean checkout also needs GNU Make and gfortran to compile the selected solver on first use.


In [ ]:
from pathlib import Path
SEARCH_ROOT = Path.cwd().resolve()
REPOSITORY = next(candidate for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents) if (candidate / 'validation' / 'pyproject.toml').is_file())
%pip install -q -e {REPOSITORY / 'validation'}


In [ ]:
import os
from avac4qgis_validation import validation_case
case = validation_case('AVAC', 'Kerswell_Coulomb')
CORES = max(1, os.cpu_count() or 1)
case.path


## Physical and numerical definition

A one-meter-deep column spreads on a horizontal bed with basal Coulomb coefficient $\mu=0.1$. The state is uniform across a five-cell wall-bounded strip. There is no Voellmy turbulent term and no fitted stopping threshold; arrest must follow from the Coulomb/static-yield implementation. Three AMR levels refine narrow corridors around the independently known boundary paths from a 40 mm base mesh to a 2.5 mm finest mesh while leaving distant regions coarse.


In [ ]:
BASE_DX_M = 0.04
AMR_LEVELS = 3
AMR_RATIO = 4
SPEED_TOLERANCE_M_S = 0.02
T_FINAL_S = 10.0
OUTPUT_FRAMES = 40
RUN_NAME = 'publication_amr'
case.run('run_avac_validation.py', '--dx', BASE_DX_M, '--t-final', T_FINAL_S, '--nout', OUTPUT_FRAMES, '--cores', CORES, '--case-name', RUN_NAME, '--amr-levels', AMR_LEVELS, '--amr-ratio', AMR_RATIO, '--speed-tolerance', SPEED_TOLERANCE_M_S, '--max1d', 1000)


## Quantitative diagnostics

Front/rear errors, arrested-state measures, mass history, and solver identity are written by the same run.


In [ ]:
summary = case.json(f'{RUN_NAME}/results/summary.json')
summary


## Comparison with Kerswell theory


In [ ]:
case.show(f'{RUN_NAME}/figures/figure_8_7_avac_vs_theory.png', f'{RUN_NAME}/figures/figure_8_10_avac_vs_theory.png', f'{RUN_NAME}/figures/profiles_avac_vs_theory.png', f'{RUN_NAME}/figures/avac_arrest_and_mass.png')
